# Generar tripletas y embeddings

In [1]:
# -*- coding: utf-8 -*-
import os
import torch
import numpy as np
import pandas as pd
import sys
import json
from types import SimpleNamespace

# Ajusta esto si es necesario para importar tu modelo TuckER
sys.path.append('../..')
# O bien apunta a la ruta absoluta donde está model.py
sys.path.append(r"C:\Users\56946\TuckER") 
from model import TuckER

# ============================================================
# 🔹 CONFIGURACIÓN (3 COHORTES - MULTIRRELACIONAL)
# ============================================================

# Carpeta del Grafo (Dataset 2019+2020+2021 Multirrelacional)
# ⚠️ Usamos la carpeta que creamos en el paso anterior
data_dir = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_multirelacional_4d"

# Carpeta de Salida
output_dir = r"C:\Users\56946\TuckER\notebooks\Experimento_warm_start\embeddings_5d_multirelacional"
os.makedirs(output_dir, exist_ok=True)

# Archivos con notas (Historia: 2019-1, 2020-1, 2021-1)
base_path = r"C:/Users/56946/TuckER/mis_scripts/dataframes_por_semestre"
csv_20191 = os.path.join(base_path, "df_20191.csv")
csv_20201 = os.path.join(base_path, "df_20201.csv")
csv_20211 = os.path.join(base_path, "df_20211.csv")

# Archivo de Puntajes
path_puntajes = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Cursos fundamentales (4 dimensiones)
cursos_1er_semestre = ["MA1101", "MA1001", "FI1000", "BT1211"]

# Dimensiones del modelo (5 para incluir el puntaje)
edim, rdim = 5, 10
kwargs = {"input_dropout": 0.2, "hidden_dropout1": 0.2, "hidden_dropout2": 0.3}

# ============================================================
# 🔹 FUNCIONES AUXILIARES
# ============================================================

def leer_y_normalizar(path):
    if not os.path.exists(path):
        print(f"⚠️ Archivo no encontrado: {path}")
        return pd.DataFrame(columns=["ID", "CURSO", "NOTA"])
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip().str.upper()
    df["ID"] = df["ID"].astype(str).str.strip().str.upper()
    df["CURSO"] = df["CURSO"].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    entities = set()
    relations = set()
    nombres = ["train.txt", "valid.txt", "test.txt", "train_balanceado.txt"]
    
    for file_name in nombres:
        path = os.path.join(data_dir, file_name)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 3:
                    h, r, t = parts[:3]
                    entities.add(h); entities.add(t); relations.add(r)
    
    relations_with_reverse = sorted(list(relations)) + [r + "_reverse" for r in sorted(list(relations))]
    return sorted(list(entities)), sorted(list(relations_with_reverse))

def heads_desde_tripletas(data_dir):
    heads = set()
    nombres = ["train.txt", "valid.txt", "test.txt"]
    for fname in nombres:
        path = os.path.join(data_dir, fname)
        if not os.path.exists(path): continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 1:
                    heads.add(parts[0])
    return heads

# ============================================================
# 🔹 1. CARGAR Y NORMALIZAR NOTAS (Rango -1 a 1)
# ============================================================

print("Cargando notas académicas (2019-1, 2020-1, 2021-1)...")
df_19 = leer_y_normalizar(csv_20191)
df_20 = leer_y_normalizar(csv_20201)
df_21 = leer_y_normalizar(csv_20211)

# Concatenar 3 generaciones
df_total = pd.concat([df_19, df_20, df_21], ignore_index=True)

# Identificar universo de alumnos con notas S1
ids_con_historia = set(df_total["ID"].unique())
print(f"   Total alumnos con notas S1: {len(ids_con_historia)}")

df_total['NOTA'] = pd.to_numeric(df_total['NOTA'], errors='coerce')

# Normalización Notas: ((n-4)/3) -> [-1, 1]
def escalar_nota(n):
    if pd.isna(n): return -1.0
    return (n - 4.0) / 3.0

df_total['NOTA'] = df_total['NOTA'].apply(escalar_nota)

def obtener_notas(df, cursos):
    # Pivot: ID x Cursos
    pivot = df[df["CURSO"].isin(cursos)].pivot_table(
        index="ID", columns="CURSO", values="NOTA", aggfunc="first"
    )
    return pivot.reindex(columns=cursos).fillna(-1.0)

notas_s1_matrix = obtener_notas(df_total, cursos_1er_semestre)

# ============================================================
# 🔹 2. CARGAR Y NORMALIZAR PUNTAJES (Rango -1 a 1)
# ============================================================

print("Cargando y normalizando puntajes...")
if os.path.exists(path_puntajes):
    df_puntajes = pd.read_csv(path_puntajes, sep=";")
    df_puntajes.columns = df_puntajes.columns.str.strip().str.upper()
    df_puntajes["ID"] = df_puntajes["ID"].astype(str).str.strip().str.upper()
    df_puntajes["PUNTAJE_PONDERADO"] = pd.to_numeric(df_puntajes["PUNTAJE_PONDERADO"], errors='coerce')
    
    # Calcular min/max usando los alumnos presentes en nuestras notas S1 (19+20+21)
    scores_gen = df_puntajes[df_puntajes["ID"].isin(ids_con_historia)]["PUNTAJE_PONDERADO"].dropna()
    
    min_score = scores_gen.min() if not scores_gen.empty else 450.0
    max_score = scores_gen.max() if not scores_gen.empty else 850.0
    
    print(f"   Rango Puntaje detectado (3 Cohortes): [{min_score}, {max_score}]")
    
    mapa_puntajes = {}
    
    # Función MinMax a [-1, 1]
    def minmax_scale(val, min_v, max_v):
        if max_v == min_v: return 0.0
        return 2 * (val - min_v) / (max_v - min_v) - 1

    for _, row in df_puntajes.iterrows():
        uid = row["ID"]
        score = row["PUNTAJE_PONDERADO"]
        
        if pd.isna(score):
            val_norm = -1.0 
        else:
            val_norm = minmax_scale(score, min_score, max_score)
            
        mapa_puntajes[uid] = val_norm
else:
    print(f"⚠️ NO SE ENCONTRÓ EL ARCHIVO DE PUNTAJES: {path_puntajes}")
    mapa_puntajes = {}

# ============================================================
# 🔹 3. VOCABULARIO DEL MODELO (3 COHORTES)
# ============================================================

if not os.path.exists(data_dir):
    print(f"❌ Error: No existe el directorio de tripletas: {data_dir}")
    sys.exit()

entities, relations = get_vocab_from_data_dir(data_dir)
print(f"✅ Vocabulario Dataset Multirrelacional: {len(entities)} entidades, {len(relations)} relaciones.")

d = SimpleNamespace()
d.entities = entities
d.relations = relations
d.entity_idxs = {e: i for i, e in enumerate(entities)}
d.relation_idxs = {r: i for i, r in enumerate(relations)}

# ============================================================
# 🔹 4. ASIGNAR EMBEDDINGS (Dim 5)
# ============================================================

modelo = TuckER(d, edim, rdim, **kwargs)
alumnos_tripletas = sorted([h for h in heads_desde_tripletas(data_dir) if h in d.entity_idxs])

contador_inicializados = 0
contador_sin_datos = 0

print(f"Inyectando vectores de dimensión {edim} (4 Notas + 1 Puntaje)...")

with torch.no_grad():
    for alumno in alumnos_tripletas:
        # 1. Obtener Notas (Dim 4)
        if alumno in notas_s1_matrix.index:
            notas_vec = notas_s1_matrix.loc[alumno].values.astype(np.float32)
            contador_inicializados += 1
        else:
            # Si no está en el registro S1, se llena con -1
            notas_vec = np.full(len(cursos_1er_semestre), -1.0, dtype=np.float32)
            contador_sin_datos += 1

        # 2. Obtener Puntaje (Dim 1)
        puntaje_val = mapa_puntajes.get(alumno, -1.0)
        
        # 3. Concatenar -> Vector final de 5 elementos
        vector_final = np.append(notas_vec, puntaje_val)
        
        # 4. Asignar al tensor
        idx = d.entity_idxs[alumno]
        modelo.E.weight[idx, :len(vector_final)] = torch.tensor(vector_final, dtype=torch.float32)

print(f"\n📊 Resumen:")
print(f"   Alumnos con historia (S1) : {contador_inicializados}")
print(f"   Alumnos sin historia (Cold): {contador_sin_datos}")

# ============================================================
# 🔹 5. GUARDAR
# ============================================================

embeddings_path = os.path.join(output_dir, "embeddings_inicializados_multi_5d.pt")
vocab_path = os.path.join(output_dir, "vocabulario_multi_5d.json")

torch.save(modelo.E.weight.data, embeddings_path)
with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump({"entities": d.entities, "relations": d.relations}, f, indent=2, ensure_ascii=False)

print(f"\n💾 Embeddings 5D guardados en: {embeddings_path}")
print(f"💾 Vocabulario guardado en: {vocab_path}")
print("✅ Listo para entrenar TuckER (3 Cohortes 5D Multirrelacional).")

Cargando notas académicas (2019-1, 2020-1, 2021-1)...
   Total alumnos con notas S1: 4155
Cargando y normalizando puntajes...
   Rango Puntaje detectado (3 Cohortes): [534.85, 935.2]
✅ Vocabulario Dataset Multirrelacional: 2368 entidades, 8 relaciones.
Inyectando vectores de dimensión 5 (4 Notas + 1 Puntaje)...

📊 Resumen:
   Alumnos con historia (S1) : 2360
   Alumnos sin historia (Cold): 0

💾 Embeddings 5D guardados en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\embeddings_5d_multirelacional\embeddings_inicializados_multi_5d.pt
💾 Vocabulario guardado en: C:\Users\56946\TuckER\notebooks\Experimento_warm_start\embeddings_5d_multirelacional\vocabulario_multi_5d.json
✅ Listo para entrenar TuckER (3 Cohortes 5D Multirrelacional).


# Generar redes

In [3]:
# -*- coding: utf-8 -*-
import os, sys, json
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from types import SimpleNamespace

# =========================
# ⚙️ CONFIGURACIÓN 5D
# =========================
DEVICE = torch.device("cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

# 1. Vocabulario (Usamos la carpeta del dataset multirrelacional)
TUCKER_DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_multirelacional_4d"

# 2. Resultados TuckER (Donde están los pesos entrenados)
RESULTS_BASE = r"C:\Users\56946\TuckER\results"

# ⚠️ IMPORTANTE: Ajusta este nombre al de tu carpeta de resultados 5D en 'results'
# Ejemplo: "Experimento_warm_start_rdim{rdim}_1000epochs..._multirelacional_5d"
RUN_PREFIX = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_multirelacional_patience400_5d_2"

# 3. Datos y Salida
BASE_DF_PATH  = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Carpeta donde guardaremos los predictores .pt
SAVE_BASE = r"C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_5d"

# Cursos
CURSOS_PRIMER     = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO    = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PERMITIDOS = CURSOS_PRIMER + CURSOS_SEGUNDO

RDIMS = list(range(1, 17)) 

# =========================
# 🛠️ FUNCIONES
# =========================
def cargar_df_notas(path_csv):
    if not os.path.exists(path_csv): 
        print(f"⚠️ Falta archivo: {path_csv}")
        return pd.DataFrame()
    df = pd.read_csv(path_csv, sep=';')
    df['ID']    = df['ID'].astype(str).str.strip().str.upper()
    df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()
    return df

def get_vocab_from_data_dir(data_dir):
    entities = set()
    nombres = ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']
    for part in nombres:
        path = os.path.join(data_dir, part)
        if not os.path.exists(path): continue
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                parts = line.strip().split()
                if len(parts) >= 1: entities.add(parts[0]); entities.add(parts[2])
    return sorted(list(entities))

def pick_state_dict(ckpt_loaded):
    if isinstance(ckpt_loaded, dict):
        if "model_state_dict" in ckpt_loaded: return ckpt_loaded["model_state_dict"]
        if "state_dict" in ckpt_loaded: return ckpt_loaded["state_dict"]
    return ckpt_loaded

def cargar_E_weights(path_pt):
    sd = pick_state_dict(torch.load(path_pt, map_location=DEVICE))
    return sd["E.weight"].detach().cpu()

# --- Normalización ---
def norm_nota(n):
    # (n - 4) / 3 -> [-1, 1]
    try:
        if pd.isna(n): return -1.0
        val = float(str(n).replace(",", "."))
        return (val - 4.0) / 3.0
    except: return -1.0

def norm_puntaje(val, min_v, max_v):
    # MinMax -> [-1, 1]
    if pd.isna(val): return -1.0
    if max_v == min_v: return 0.0
    return 2 * (val - min_v) / (max_v - min_v) - 1

def construir_vectores_5d(df_sem1, df_sem2, df_ptje, cursos_primer, cursos_permitidos):
    # 1. Filtro: Alumnos con 4 cursos en S1 y algo en S2
    df_fund = df_sem1[df_sem1['CURSO'].isin(cursos_primer)]
    conteo = df_fund.groupby('ID')['CURSO'].nunique()
    alumnos_4 = conteo[conteo == len(cursos_primer)].index

    df_sem2_filt = df_sem2[
        (df_sem2['ID'].isin(alumnos_4)) &
        (df_sem2['CURSO'].isin(cursos_permitidos))
    ]
    alumnos_validos = sorted(df_sem2_filt['ID'].unique())

    if not alumnos_validos: return pd.DataFrame()

    # 2. Matriz de Notas
    df_notas = df_sem1[df_sem1['ID'].isin(alumnos_validos) & df_sem1['CURSO'].isin(cursos_primer)].copy()
    
    pivot = df_notas.pivot_table(index="ID", columns="CURSO", values="NOTA", aggfunc="first")
    pivot = pivot.reindex(columns=cursos_primer)
    pivot = pivot.applymap(norm_nota)
    
    # 3. Agregar Puntaje
    ptjes_validos = df_ptje[df_ptje["ID"].isin(alumnos_validos)]["PUNTAJE_PONDERADO"].dropna()
    min_s = ptjes_validos.min() if not ptjes_validos.empty else 450.0
    max_s = ptjes_validos.max() if not ptjes_validos.empty else 850.0
    
    mapa_ptje = df_ptje.set_index("ID")["PUNTAJE_PONDERADO"].to_dict()
    
    col_ptje = []
    for uid in pivot.index:
        raw_p = mapa_ptje.get(uid, np.nan)
        col_ptje.append(norm_puntaje(raw_p, min_s, max_s))
        
    pivot["PUNTAJE"] = col_ptje
    return pivot

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def entrenar_predictor(X_data, Y_data, save_path, max_epochs=500, lr=1e-3, patience=100):
    X_train, X_test, Y_train, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)
    X_tr, X_val, Y_tr, Y_val          = train_test_split(X_train, Y_train, test_size=0.15, random_state=42)

    X_tr_t = torch.FloatTensor(X_tr); Y_tr_t = torch.FloatTensor(Y_tr)
    X_val_t= torch.FloatTensor(X_val);Y_val_t= torch.FloatTensor(Y_val)
    X_te_t = torch.FloatTensor(X_test);Y_te_t= torch.FloatTensor(Y_test)

    model = EmbeddingPredictor(X_tr.shape[1], Y_tr.shape[1]).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val, patience_counter = float('inf'), 0

    for epoch in range(1, max_epochs + 1):
        model.train(); optimizer.zero_grad()
        loss = criterion(model(X_tr_t), Y_tr_t)
        loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad(): vloss = criterion(model(X_val_t), Y_val_t)

        if vloss.item() < best_val - 1e-9:
            best_val = vloss.item()
            torch.save(model.state_dict(), save_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience: break

    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    model.eval()
    with torch.no_grad(): test_mse = criterion(model(X_te_t), Y_te_t).item()
    return test_mse

# =========================
# MAIN
# =========================
def main():
    if not os.path.exists(SAVE_BASE): os.makedirs(SAVE_BASE)
    print(f"📂 Guardando redes en: {SAVE_BASE}")
    print("=== Entrenando Redes MULTIRRELACIONALES 5D (3 Cohortes) ===")

    # 1. Vocabulario
    vocab = get_vocab_from_data_dir(TUCKER_DATA_DIR)
    entity_idxs = {e: i for i, e in enumerate(vocab)}
    
    # 2. Cargar Datos
    df_19_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20191.csv"))
    df_19_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20192.csv"))
    df_20_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20201.csv"))
    df_20_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20202.csv"))
    df_21_1 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20211.csv"))
    df_21_2 = cargar_df_notas(os.path.join(BASE_DF_PATH, "df_20212.csv"))
    
    df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
    df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')

    # 3. Construir Vectores 5D (Unión de 3 generaciones)
    print("   Construyendo vectores 5D (19+20+21)...")
    vecs_19 = construir_vectores_5d(df_19_1, df_19_2, df_ptje, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    vecs_20 = construir_vectores_5d(df_20_1, df_20_2, df_ptje, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    vecs_21 = construir_vectores_5d(df_21_1, df_21_2, df_ptje, CURSOS_PRIMER, CURSOS_PERMITIDOS)
    
    X_total_df = pd.concat([vecs_19, vecs_20, vecs_21], axis=0)
    X_total_df = X_total_df[~X_total_df.index.duplicated(keep='first')]
    
    print(f"-> Vectores X listos: {len(X_total_df)} alumnos. Dim={X_total_df.shape[1]}")

    # 4. Entrenar
    resumen = []
    for rdim in RDIMS:
        # Busca el TuckER
        run_dir   = os.path.join(RESULTS_BASE, RUN_PREFIX.format(rdim=rdim))
        tucker_pt = os.path.join(run_dir, "best_model.pt")
        save_path = os.path.join(SAVE_BASE, f"best_predictor_rdim{rdim}_multi_5d.pt")

        print(f"\n>> rdim={rdim}")
        if not os.path.exists(tucker_pt):
            print(f"   ⚠️ Pendiente/No existe: {tucker_pt}")
            continue

        try:
            E = cargar_E_weights(tucker_pt)
            
            alumnos_comunes = sorted(set(X_total_df.index).intersection(entity_idxs.keys()))
            if not alumnos_comunes:
                print("   ⚠️ Sin intersección.")
                continue
                
            X_data = X_total_df.loc[alumnos_comunes].values.astype(np.float32)
            idxs   = [entity_idxs[a] for a in alumnos_comunes]
            Y_data = E[idxs].numpy()
            
            mse = entrenar_predictor(X_data, Y_data, save_path)
            print(f"   ✅ Guardado. MSE: {mse:.6f}")
            resumen.append((rdim, mse))
            
        except Exception as e:
            print(f"   ❌ Error: {e}")

    if resumen:
        pd.DataFrame(resumen, columns=["rdim", "mse"]).to_csv(os.path.join(SAVE_BASE, "resumen_multi_5d.csv"), index=False)
        print("\n✅ Proceso finalizado.")

if __name__ == "__main__":
    main()

📂 Guardando redes en: C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_5d
=== Entrenando Redes MULTIRRELACIONALES 5D (3 Cohortes) ===
   Construyendo vectores 5D (19+20+21)...
-> Vectores X listos: 2351 alumnos. Dim=5

>> rdim=1
   ✅ Guardado. MSE: 0.008288

>> rdim=2
   ✅ Guardado. MSE: 0.007833

>> rdim=3
   ✅ Guardado. MSE: 0.009634

>> rdim=4
   ✅ Guardado. MSE: 0.012582

>> rdim=5
   ✅ Guardado. MSE: 0.019558

>> rdim=6
   ✅ Guardado. MSE: 0.013585

>> rdim=7
   ✅ Guardado. MSE: 0.006698

>> rdim=8
   ✅ Guardado. MSE: 0.009377

>> rdim=9
   ✅ Guardado. MSE: 0.021044

>> rdim=10
   ✅ Guardado. MSE: 0.010088

>> rdim=11
   ✅ Guardado. MSE: 0.007729

>> rdim=12
   ✅ Guardado. MSE: 0.006712

>> rdim=13
   ✅ Guardado. MSE: 0.012050

>> rdim=14
   ✅ Guardado. MSE: 0.008456

>> rdim=15
   ✅ Guardado. MSE: 0.010720

>> rdim=16
   ✅ Guardado. MSE: 0.012818

✅ Proceso finalizado.


# Probando modelo data augmentation-multirelacional

In [5]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN 5D (3 COHORTES)
# ============================================================

# Vocabulario del Dataset Multirrelacional
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_multirelacional_4d"

# Ruta base de resultados TuckER
RESULTS_BASE = r"C:\Users\56946\TuckER\results"
# Prefijo de la carpeta del experimento TuckER (Ajustado a tu config)
RUN_PREFIX = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_multirelacional_patience400_5d_2"

# Ruta de los Predictores Neuronales (.pt)
PRED_DIR_BASE = r"C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_5d"

# Datos Evaluación
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

# Evaluamos en 2021 (que era parte del test set global, pero validamos la inferencia aquí)
CSV_S1 = os.path.join(BASE_PATH, "df_20211.csv")
CSV_S2 = os.path.join(BASE_PATH, "df_20212.csv")

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]
RDIMS = range(1, 17) 
DEVICE = "cpu"

# ============================================================
# 🔹 MODELOS
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    """
    Busca el diccionario de pesos dentro del checkpoint.
    Si no encuentra 'model_state_dict' ni 'state_dict', asume que 'ckpt' ya son los pesos.
    """
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
        # ⚠️ FIX: Si no tiene esas llaves, asumimos que el diccionario YA ES el state_dict
        return ckpt
    return ckpt

def load_predictor(path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(pick_state_dict(state))
    model.to(DEVICE).eval()
    return model

def load_tucker_weights(path):
    raw = torch.load(path, map_location=DEVICE)
    state = pick_state_dict(raw)
    return state["E.weight"].to(DEVICE), state["R.weight"].to(DEVICE), state["W"].to(DEVICE)

def get_vocab(data_dir):
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h); entities.add(t); relations.add(r)
    return SimpleNamespace(
        entities=sorted(list(entities)),
        relations=sorted(list(relations) + [r+"_reverse" for r in relations]),
        entity_idxs={e:i for i,e in enumerate(sorted(list(entities)))},
        relation_idxs={r:i for i,r in enumerate(sorted(list(relations) + [r+"_reverse" for r in relations]))}
    )

def find_rel_idx(vocab, hint):
    cands = [r for r in vocab.relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return vocab.relation_idxs[cands[0]]

@torch.no_grad()
def tucker_score(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    else: # Layout alternativo
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ============================================================
# 🔹 PROCESAMIENTO DE DATOS 5D (NOTAS + PUNTAJE)
# ============================================================

def norm_nota(n):
    # (n - 4) / 3 -> [-1, 1]
    if pd.isna(n): return -1.0
    val = float(str(n).replace(",", "."))
    return (val - 4.0) / 3.0

def norm_puntaje(val, min_v, max_v):
    if pd.isna(val): return -1.0
    if max_v == min_v: return 0.0
    return 2 * (val - min_v) / (max_v - min_v) - 1

def get_input_vector_5d(df_s1, df_ptje, aid, cursos_primer, min_s, max_s):
    # 1. Notas
    subset = df_s1[(df_s1['ID'] == aid) & (df_s1['CURSO'].isin(cursos_primer))]
    vec = [-1.0] * len(cursos_primer)
    
    # Mapeo rápido curso->nota
    if not subset.empty:
        notas_dict = dict(zip(subset['CURSO'], subset['NOTA']))
        for i, c in enumerate(cursos_primer):
            if c in notas_dict:
                vec[i] = norm_nota(notas_dict[c])
    
    # 2. Puntaje
    row_p = df_ptje[df_ptje["ID"] == aid]
    if not row_p.empty:
        raw_p = row_p.iloc[0]["PUNTAJE_PONDERADO"]
        p_val = norm_puntaje(raw_p, min_s, max_s)
    else:
        p_val = -1.0
        
    vec.append(p_val)
    return torch.tensor(vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- EVALUACIÓN FINAL 5D (3 COHORTES) ---")
    
    # Cargar DF
    df_s1 = pd.read_csv(CSV_S1, sep=';')
    df_s2 = pd.read_csv(CSV_S2, sep=';')
    for df in (df_s1, df_s2):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    # Cargar Puntajes y calcular min/max global
    df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
    df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')
    min_s = df_ptje["PUNTAJE_PONDERADO"].min()
    max_s = df_ptje["PUNTAJE_PONDERADO"].max()

    vocab = get_vocab(DATA_DIR)
    
    # Preparar datos de evaluación
    alumnos_validos = df_s1.groupby("ID")["CURSO"].apply(set)
    ids_ok = alumnos_validos[alumnos_validos.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index
    
    df_eval = df_s2[
        (df_s2['ID'].isin(ids_ok)) & 
        (df_s2['CURSO'].isin(CURSOS_PRED_ALL))
    ].copy()
    
    def get_real_rel(row):
        n = pd.to_numeric(row['NOTA'], errors='coerce')
        st = str(row['ESTADO_CURSO'])
        if "Reprobado" in st or pd.isna(n) or n < 4.0: return "reprueba"
        if n < 5.0: return "aprueba_4_5"
        if n < 6.0: return "aprueba_5_6"
        return "aprueba_6_7"
    
    df_eval["REL_REAL"] = df_eval.apply(get_real_rel, axis=1)
    
    print(f"Total Evaluaciones: {len(df_eval)}")

    for rdim in RDIMS:
        print(f"\n>> Evaluando rdim={rdim}")
        
        # Rutas dinámicas
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        pred_pt = os.path.join(PRED_DIR_BASE, f"best_predictor_rdim{rdim}_multi_5d.pt")
        
        if not os.path.exists(tucker_pt) or not os.path.exists(pred_pt):
            print("   ⚠️ Faltan archivos, saltando...")
            continue
            
        E, R, W = load_tucker_weights(tucker_pt)
        d1 = E.shape[1]
        
        # Mapear relaciones
        rel_map = {}
        for rname in RELACIONES:
            idx = find_rel_idx(vocab, rname)
            if idx is not None: rel_map[rname] = idx
            
        predictor = load_predictor(pred_pt, input_size=5, out_dim=d1) # Input 5D!
        
        # Cache embeddings
        ehat_cache = {}
        unique_ids = df_eval["ID"].unique()
        for uid in unique_ids:
            x = get_input_vector_5d(df_s1, df_ptje, uid, CURSOS_PRIMER, min_s, max_s)
            with torch.no_grad(): ehat_cache[uid] = predictor(x).squeeze(0)
            
        # Evaluar
        y_true, y_pred = [], []
        
        for _, row in df_eval.iterrows():
            uid, cur, rel_real = row["ID"], row["CURSO"], row["REL_REAL"]
            if cur not in vocab.entity_idxs: continue
            
            t_idx = vocab.entity_idxs[cur]
            e_hat = ehat_cache[uid]
            
            scores = {r: tucker_score(e_hat, R, W, E, idx, t_idx) for r, idx in rel_map.items()}
            rel_pred = max(scores, key=scores.get)
            
            # Binarizar: Riesgo = Reprueba
            y_true.append(1 if "reprueba" in rel_real else 0)
            y_pred.append(1 if "reprueba" in rel_pred else 0)
            
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        recall = tp / (tp + fn) if (tp+fn) > 0 else 0
        prec = tp / (tp + fp) if (tp+fp) > 0 else 0
        acc = (tp + tn) / len(y_true)
        
        print("-" * 50)
        print(f"Riesgo Real (TP+FN): {tp+fn}")
        print(f"Recall:    {recall:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Matriz: TP={tp}, FN={fn}, FP={fp}, TN={tn}")

if __name__ == "__main__":
    main()


--- EVALUACIÓN FINAL 5D (3 COHORTES) ---
Total Evaluaciones: 3023

>> Evaluando rdim=1
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.4506
Precision: 0.0726
Accuracy:  0.5141
Matriz: TP=105, FN=128, FP=1341, TN=1449

>> Evaluando rdim=2
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.0000
Precision: 0.0000
Accuracy:  0.9229
Matriz: TP=0, FN=233, FP=0, TN=2790

>> Evaluando rdim=3
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.0000
Precision: 0.0000
Accuracy:  0.9229
Matriz: TP=0, FN=233, FP=0, TN=2790

>> Evaluando rdim=4
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.0043
Precision: 1.0000
Accuracy:  0.9233
Matriz: TP=1, FN=232, FP=0, TN=2790

>> Evaluando rdim=5
--------------------------------------------------
Riesgo Real (TP+FN): 233
Recall:    0.2403
Precision: 0.0783
Accuracy:  0.7235
Matriz: TP=56, FN=177, FP=659

# colapsando aprueba con 4,5 y reprueba

In [6]:
# -*- coding: utf-8 -*-
import os, re, sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from types import SimpleNamespace

# ============================================================
# 🔹 CONFIGURACIÓN 5D (3 COHORTES)
# ============================================================

# Vocabulario
DATA_DIR = r"C:\Users\56946\TuckER\data\dataset_2019_2020_2021_multirelacional_4d"

# Rutas TuckER
RESULTS_BASE = r"C:\Users\56946\TuckER\results"
# ⚠️ Ajusta este nombre si tu carpeta de resultados TuckER tiene otro sufijo
RUN_PREFIX = r"Experimento_warm_start_rdim{rdim}_1000epochs_earlystopping_2019_2020_2021_multirelacional_patience400_5d_2"

# Ruta Predictores 5D
PRED_DIR_BASE = r"C:/Users/56946/TuckER/notebooks/Experimento_warm_start_multirelacional/redes_3cohortes_multirelacional_5d"

# Datos
BASE_PATH = r"C:\Users\56946\TuckER\mis_scripts\dataframes_por_semestre"
CSV_S1 = os.path.join(BASE_PATH, "df_20211.csv")
CSV_S2 = os.path.join(BASE_PATH, "df_20212.csv")
RUTA_PUNTAJES = r"C:\Users\56946\TuckER\mis_scripts\new_df_por_sem\base_puntaje_estudiante.csv"

CURSOS_PRIMER   = ['MA1101','MA1001','FI1000','BT1211']
CURSOS_SEGUNDO  = ['MA1002','MA1102','FI1100','CC1002']
CURSOS_PRED_ALL = CURSOS_PRIMER + CURSOS_SEGUNDO

RELACIONES = ["reprueba", "aprueba_4_5", "aprueba_5_6", "aprueba_6_7"]
RDIMS = range(1, 17) 
DEVICE = "cpu"

# ============================================================
# 🔹 MODELOS Y UTILIDADES
# ============================================================

class EmbeddingPredictor(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, output_size)
        )
    def forward(self, x): return self.network(x)

def pick_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "model_state_dict" in ckpt: return ckpt["model_state_dict"]
        if "state_dict" in ckpt: return ckpt["state_dict"]
        return ckpt
    return ckpt

def load_predictor(path, input_size, out_dim):
    model = EmbeddingPredictor(input_size, out_dim)
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(pick_state_dict(state))
    model.to(DEVICE).eval()
    return model

def load_tucker_weights(path):
    raw = torch.load(path, map_location=DEVICE)
    state = pick_state_dict(raw)
    return state["E.weight"].to(DEVICE), state["R.weight"].to(DEVICE), state["W"].to(DEVICE)

def get_vocab(data_dir):
    entities, relations = set(), set()
    for part in ['train.txt', 'valid.txt', 'test.txt', 'train_balanceado.txt']:
        path = os.path.join(data_dir, part)
        if os.path.exists(path):
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip(): continue
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        h, r, t = parts[:3]
                        entities.add(h); entities.add(t); relations.add(r)
    return SimpleNamespace(
        entities=sorted(list(entities)),
        relations=sorted(list(relations) + [r+"_reverse" for r in relations]),
        entity_idxs={e:i for i,e in enumerate(sorted(list(entities)))},
        relation_idxs={r:i for i,r in enumerate(sorted(list(relations) + [r+"_reverse" for r in relations]))}
    )

def find_rel_idx(vocab, hint):
    cands = [r for r in vocab.relations if re.search(hint, r, re.I)]
    if not cands: return None
    cands.sort(key=lambda x: (("_reverse" in x), x))
    return vocab.relation_idxs[cands[0]]

@torch.no_grad()
def tucker_score(e_hat, R, W, E, r_idx, t_idx):
    d1 = E.shape[1]
    r = R[r_idx]
    if W.shape[0] == r.numel() and W.shape[1] == d1:
        M_r = torch.tensordot(W, r, dims=([0],[0])) 
    else:
        M_r = torch.tensordot(W, r, dims=([1],[0]))
    v = e_hat @ M_r
    e_t = E[t_idx]
    return float(torch.dot(v, e_t))

# ============================================================
# 🔹 CONSTRUCCIÓN DE VECTORES (5D)
# ============================================================
def norm_nota(n):
    # (n - 4) / 3 -> [-1, 1]
    if pd.isna(n): return -1.0
    try:
        val = float(str(n).replace(",", "."))
        return (val - 4.0) / 3.0
    except: return -1.0

def norm_puntaje(val, min_v, max_v):
    if pd.isna(val): return -1.0
    if max_v == min_v: return 0.0
    return 2 * (val - min_v) / (max_v - min_v) - 1

def get_input_vector_5d(df_s1, df_ptje, aid, cursos_primer, min_s, max_s):
    # 1. Notas
    subset = df_s1[(df_s1['ID'] == aid) & (df_s1['CURSO'].isin(cursos_primer))]
    vec = [-1.0] * len(cursos_primer)
    if not subset.empty:
        notas_dict = dict(zip(subset['CURSO'], subset['NOTA']))
        for i, c in enumerate(cursos_primer):
            if c in notas_dict:
                vec[i] = norm_nota(notas_dict[c])
    
    # 2. Puntaje
    row_p = df_ptje[df_ptje["ID"] == aid]
    if not row_p.empty:
        raw_p = row_p.iloc[0]["PUNTAJE_PONDERADO"]
        p_val = norm_puntaje(raw_p, min_s, max_s)
    else:
        p_val = -1.0
    
    vec.append(p_val)
    return torch.tensor(vec, dtype=torch.float32).view(1, -1)

# ============================================================
# 🔹 LÓGICA RIESGO AMPLIADO
# ============================================================
def es_riesgo_ampliado(relacion_str):
    rel = relacion_str.lower()
    # Riesgo = Reprueba o Aprueba [4.0, 5.0)
    return "reprueba" in rel or "aprueba_4_5" in rel

# ============================================================
# 🔹 MAIN
# ============================================================
def main():
    print("\n--- EVALUACIÓN FINAL 5D (RIESGO AMPLIADO) ---")
    
    df_s1 = pd.read_csv(CSV_S1, sep=';')
    df_s2 = pd.read_csv(CSV_S2, sep=';')
    for df in (df_s1, df_s2):
        df['ID'] = df['ID'].astype(str).str.strip().str.upper()
        df['CURSO'] = df['CURSO'].astype(str).str.strip().str.upper()

    df_ptje = pd.read_csv(RUTA_PUNTAJES, sep=';')
    df_ptje["ID"] = df_ptje["ID"].astype(str).str.strip().str.upper()
    df_ptje["PUNTAJE_PONDERADO"] = pd.to_numeric(df_ptje["PUNTAJE_PONDERADO"], errors='coerce')
    min_s = df_ptje["PUNTAJE_PONDERADO"].min()
    max_s = df_ptje["PUNTAJE_PONDERADO"].max()

    vocab = get_vocab(DATA_DIR)
    
    alumnos_s1 = df_s1.groupby("ID")["CURSO"].apply(set)
    ids_ok = alumnos_s1[alumnos_s1.apply(lambda s: set(CURSOS_PRIMER).issubset(s))].index
    
    df_eval = df_s2[
        (df_s2['ID'].isin(ids_ok)) & 
        (df_s2['CURSO'].isin(CURSOS_PRED_ALL))
    ].copy()
    
    def get_real_rel(row):
        n = pd.to_numeric(row['NOTA'], errors='coerce')
        st = str(row['ESTADO_CURSO'])
        if "Reprobado" in st or pd.isna(n) or n < 4.0: return "reprueba"
        if n < 5.0: return "aprueba_4_5"
        if n < 6.0: return "aprueba_5_6"
        return "aprueba_6_7"
    
    df_eval["REL_REAL"] = df_eval.apply(get_real_rel, axis=1)
    print(f"Evaluaciones totales: {len(df_eval)}")

    for rdim in RDIMS:
        print(f"\n>> Evaluando rdim={rdim}")
        
        tucker_folder = RUN_PREFIX.format(rdim=rdim)
        tucker_pt = os.path.join(RESULTS_BASE, tucker_folder, "best_model.pt")
        pred_pt = os.path.join(PRED_DIR_BASE, f"best_predictor_rdim{rdim}_multi_5d.pt")
        
        if not os.path.exists(tucker_pt) or not os.path.exists(pred_pt):
            print("   ⏩ Saltando (Faltan archivos)")
            continue
            
        E, R, W = load_tucker_weights(tucker_pt)
        d1 = E.shape[1]
        
        rel_map = {}
        for rname in RELACIONES:
            idx = find_rel_idx(vocab, rname)
            if idx is not None: rel_map[rname] = idx
            
        predictor = load_predictor(pred_pt, input_size=5, out_dim=d1)
        
        ehat_cache = {}
        for uid in df_eval["ID"].unique():
            x = get_input_vector_5d(df_s1, df_ptje, uid, CURSOS_PRIMER, min_s, max_s)
            with torch.no_grad(): ehat_cache[uid] = predictor(x).squeeze(0)
            
        y_true, y_pred = [], []
        
        for _, row in df_eval.iterrows():
            uid, cur, rel_real = row["ID"], row["CURSO"], row["REL_REAL"]
            if cur not in vocab.entity_idxs: continue
            
            t_idx = vocab.entity_idxs[cur]
            e_hat = ehat_cache[uid]
            
            scores = {r: tucker_score(e_hat, R, W, E, idx, t_idx) for r, idx in rel_map.items()}
            rel_pred = max(scores, key=scores.get)
            
            # ⚠️ COLAPSO RIESGO AMPLIADO
            y_true.append(1 if es_riesgo_ampliado(rel_real) else 0)
            y_pred.append(1 if es_riesgo_ampliado(rel_pred) else 0)
            
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        recall = tp / (tp + fn) if (tp+fn) > 0 else 0
        prec = tp / (tp + fp) if (tp+fp) > 0 else 0
        acc = (tp + tn) / len(y_true)
        
        print("-" * 60)
        print(f"Accuracy Global          : {acc:.4f}")
        print(f"Total REALES en Riesgo   : {tp+fn} (Repro + Nota 4-5)")
        print("-" * 60)
        print(f"✅ RECALL (Sensibilidad) : {recall:.4f}")
        print(f"   (Detectamos {tp} de {tp+fn} casos)")
        print("-" * 60)
        print(f"🎯 PRECISION             : {prec:.4f}")
        print(f"Matriz: TP={tp}, FN={fn}, FP={fp}, TN={tn}")

if __name__ == "__main__":
    main()


--- EVALUACIÓN FINAL 5D (RIESGO AMPLIADO) ---
Evaluaciones totales: 3023

>> Evaluando rdim=1
------------------------------------------------------------
Accuracy Global          : 0.4376
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.3428
   (Detectamos 277 de 808 casos)
------------------------------------------------------------
🎯 PRECISION             : 0.1916
Matriz: TP=277, FN=531, FP=1169, TN=1046

>> Evaluando rdim=2
------------------------------------------------------------
Accuracy Global          : 0.7327
Total REALES en Riesgo   : 808 (Repro + Nota 4-5)
------------------------------------------------------------
✅ RECALL (Sensibilidad) : 0.0000
   (Detectamos 0 de 808 casos)
------------------------------------------------------------
🎯 PRECISION             : 0.0000
Matriz: TP=0, FN=808, FP=0, TN=2215

>> Evaluando rdim=3
--------------------------------------------------------